# Web Server Log Analysis Using Hadoop MapReduce
### NASA Kennedy Space Center HTTP Logs (July 1995)

**Architecture**: Pure Hadoop Streaming with Python (`mappers.py` & `reducers.py`)  
**Environment**: Google Colab (Java 11 + Apache Hadoop 3.4.1)  


## 1. Setup Java 11 and Hadoop 3.4.1

In [ ]:
# Install OpenJDK 11
!apt-get update -qq && apt-get install -y -qq openjdk-11-jdk-headless

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

# Download & Extract Hadoop 3.4.1
if not os.path.exists("hadoop-3.4.1"):
    print("Downloading Hadoop 3.4.1...")
    !wget -q -N https://downloads.apache.org/hadoop/common/hadoop-3.4.1/hadoop-3.4.1.tar.gz || \
     wget -q -N https://archive.apache.org/dist/hadoop/common/hadoop-3.4.1/hadoop-3.4.1.tar.gz
    !tar -xzf hadoop-3.4.1.tar.gz

os.environ["HADOOP_HOME"] = os.path.abspath("hadoop-3.4.1")
os.environ["PATH"] = f"{os.environ['HADOOP_HOME']}/bin:{os.environ['PATH']}"
!hadoop version


## 2. Download and Extract NASA Dataset

In [ ]:
import urllib.request, gzip, shutil

os.makedirs("data/raw", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
os.makedirs("charts", exist_ok=True)

gz_path = "data/raw/NASA_access_log_Jul95.gz"
log_path = "data/raw/NASA_access_log_Jul95.log"
url = "http://ita.ee.lbl.gov/traces/NASA_access_log_Jul95.gz"

if not os.path.exists(log_path):
    if not os.path.exists(gz_path):
        print("Downloading dataset...")
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as resp, open(gz_path, "wb") as out_f:
            shutil.copyfileobj(resp, out_f)
    print("Extracting dataset...")
    with gzip.open(gz_path, "rb") as f_in, open(log_path, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
print(f"Log ready: {log_path} ({os.path.getsize(log_path):,} bytes)")


## 3. Run the 6 Hadoop Streaming Jobs
Using `src/mappers.py` and `src/reducers.py` with job arguments (`hourly`, `urls`, `status`, `ips`, `daily_bytes`, `daily_errors`).

In [ ]:
import subprocess, sys

streaming_jar = os.path.join(
    os.environ.get("HADOOP_HOME", ""),
    "share/hadoop/tools/lib/hadoop-streaming-3.4.1.jar"
)
has_hadoop = os.path.exists(streaming_jar)
print(f"Hadoop Streaming Available: {has_hadoop}")

def run_job(job_name):
    out_dir = f"outputs/{job_name}"
    if os.path.exists(out_dir):
        shutil.rmtree(out_dir)
    
    if has_hadoop:
        cmd = [
            "hadoop", "jar", streaming_jar,
            "-files", "src/mappers.py,src/reducers.py,src/log_parser.py",
            "-mapper", f"python3 mappers.py {job_name}",
            "-reducer", f"python3 reducers.py {job_name}",
            "-input", "data/raw/NASA_access_log_Jul95.log",
            "-output", out_dir,
            "-numReduceTasks", "1"
        ]
        print(f"Running Hadoop job: {job_name}...")
        subprocess.run(cmd, check=True)
    else:
        print(f"Running Streaming Simulation for {job_name}...")
        os.makedirs(out_dir, exist_ok=True)
        m_proc = subprocess.Popen([sys.executable, "src/mappers.py", job_name],
                                  stdin=open("data/raw/NASA_access_log_Jul95.log", "r", encoding="latin-1", errors="replace"),
                                  stdout=subprocess.PIPE, text=True, encoding="latin-1", errors="replace")
        m_out, _ = m_proc.communicate()
        lines = [l for l in m_out.splitlines() if l.strip()]
        lines.sort()
        r_proc = subprocess.Popen([sys.executable, "src/reducers.py", job_name],
                                  stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True, encoding="latin-1", errors="replace")
        r_out, _ = r_proc.communicate(input="\n".join(lines) + "\n")
        with open(f"{out_dir}/part-00000", "w", encoding="utf-8") as f:
            f.write(r_out)
    print(f"Finished {job_name}.")

for j in ["hourly", "urls", "status", "ips", "daily_bytes", "daily_errors"]:
    run_job(j)


## 4. Export CSV Files & Render Charts

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")

# Load DataFrames
df_hourly = pd.read_csv("outputs/hourly/part-00000", sep="\t", names=["hour", "request_count"], dtype={"hour": str, "request_count": int})
df_urls = pd.read_csv("outputs/urls/part-00000", sep="\t", names=["url", "request_count"], dtype={"request_count": int})
df_status = pd.read_csv("outputs/status/part-00000", sep="\t", names=["status_code", "request_count"], dtype={"status_code": int, "request_count": int})
df_ips = pd.read_csv("outputs/ips/part-00000", sep="\t", names=["client_host", "request_count"], dtype={"request_count": int})
df_bytes = pd.read_csv("outputs/daily_bytes/part-00000", sep="\t", names=["date", "total_bytes"], dtype={"total_bytes": int})
df_errors = pd.read_csv("outputs/daily_errors/part-00000", sep="\t", names=["date", "error_count", "total_requests", "error_rate_percent"], dtype={"error_count": int, "total_requests": int, "error_rate_percent": float})

# Visualizations
fig, axes = plt.subplots(3, 2, figsize=(15, 12))

# 1. Hourly
axes[0, 0].bar(df_hourly["hour"], df_hourly["request_count"], color="#1f77b4")
axes[0, 0].set_title("1. Hourly Request Distribution (00-23 EDT)", fontweight="bold")

# 2. Top URLs
axes[0, 1].barh(df_urls["url"].iloc[::-1], df_urls["request_count"].iloc[::-1], color="#2ca02c")
axes[0, 1].set_title("2. Top 10 Requested URLs", fontweight="bold")

# 3. Status Codes
axes[1, 0].bar(df_status["status_code"].astype(str), df_status["request_count"], color="#ff7f0e")
axes[1, 0].set_yscale("log")
axes[1, 0].set_title("3. Status Codes Distribution (Log Scale)", fontweight="bold")

# 4. Top IPs
axes[1, 1].barh(df_ips["client_host"].iloc[::-1], df_ips["request_count"].iloc[::-1], color="#9467bd")
axes[1, 1].set_title("4. Top 10 Client Hosts / IPs", fontweight="bold")

# 5. Daily Bytes
axes[2, 0].plot(df_bytes["date"], df_bytes["total_bytes"] / (1024**3), marker="o", color="#d62728")
axes[2, 0].tick_params(axis="x", rotation=45)
axes[2, 0].set_title("5. Daily Transferred Volume (GB)", fontweight="bold")

# 6. Error Rate
axes[2, 1].plot(df_errors["date"], df_errors["error_rate_percent"], marker="s", color="#e377c2")
axes[2, 1].tick_params(axis="x", rotation=45)
axes[2, 1].set_title("6. Daily HTTP Error Rate (%)", fontweight="bold")

plt.tight_layout()
plt.show()


## 5. Data Validation Audit

In [ ]:
total_valid = 1891714
assert df_hourly["request_count"].sum() == total_valid
assert df_status["request_count"].sum() == total_valid
assert df_errors["total_requests"].sum() == total_valid
print("All MapReduce sums match 1,891,714 valid records!")
